# 03 — 多路召回模型基准测试（Benchmark）

本 Notebook 对四种召回器（Top-Pop、ItemCF、BPR-MF、ALS）进行离线评估，
并通过 MultiRecall 融合验证多路召回的增益效果。

**评估指标**：HR@K、Recall@K、NDCG@K（K=10/50/100）、Coverage@100

**评估集**：validation split（按时间划分，防数据穿越）

In [ ]:
import sys
from pathlib import Path

_nb_dir = Path(__file__).parent if "__file__" in dir() else Path.cwd()
for _p in [_nb_dir / "src", _nb_dir.parent / "src"]:
    if (_p / "ecom_rec").exists():
        sys.path.insert(0, str(_p))
        break

_root = _root = _nb_dir if (_nb_dir / "data").exists() else _nb_dir.parent
PROCESSED = _root / "data" / "processed"
FIGURES   = _root / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def _save_fig(fig, fname, scale=2):
    try:
        fig.write_image(str(FIGURES / fname), scale=scale)
    except Exception as e:
        print(f"图片保存跳过（{fname}）：{e}")

import polars as pl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from ecom_rec.recall.pop import PopRecaller
from ecom_rec.recall.itemcf import ItemCFRecaller
from ecom_rec.recall.bpr import BPRRecaller
from ecom_rec.recall.als import ALSRecaller
from ecom_rec.pipeline.multi_recall import MultiRecall
from ecom_rec.eval.recall_metrics import evaluate_recall

train = pl.read_parquet(PROCESSED / "train.parquet")
valid = pl.read_parquet(PROCESSED / "valid.parquet")
test  = pl.read_parquet(PROCESSED / "test.parquet")
all_items = set(train["item_id"].to_list())

ground_truth_valid = {}
for row in (
    valid
    .group_by("user_id")
    .agg(pl.col("item_id").alias("items"))
    .iter_rows(named=True)
):
    ground_truth_valid[row["user_id"]] = row["items"].to_list()

eval_users = list(ground_truth_valid.keys())[:2000]
print(f"训练集大小：{len(train):,}")
print(f"验证集大小：{len(valid):,}")
print(f"全量商品数：{len(all_items):,}")
print(f"评估用户数：{len(eval_users):,}")

## 1. 训练四种召回模型

In [ ]:
# 逐一训练 PopRecaller, ItemCFRecaller, BPRRecaller, ALSRecaller
pop = PopRecaller().fit(train)
icf = ItemCFRecaller(n_neighbors=20).fit(train)
bpr = BPRRecaller(factors=64, iterations=50).fit(train)
als = ALSRecaller(factors=64, iterations=30).fit(train)
models = {"Top-Pop": pop, "ItemCF": icf, "BPR-MF": bpr, "ALS": als}
print("所有召回模型训练完成")

## 2. 评估各模型（k=10,50,100）

In [ ]:
# 构建 ground_truth，批量推荐，调用 evaluate_recall，格式化为 DataFrame
K_LIST = [10, 50, 100]
results = {}
for name, model in models.items():
    recs = model.recommend_batch(eval_users, k=100)
    results[name] = evaluate_recall(
        recs, ground_truth_valid, all_items, k_list=K_LIST
    )
    print(
        f"{name:10s}: "
        f"HR@50={results[name]['HR@50']:.4f}, "
        f"Recall@50={results[name]['Recall@50']:.4f}, "
        f"NDCG@50={results[name]['NDCG@50']:.4f}"
    )

# 汇总为 DataFrame
df_results = pd.DataFrame(results).T
df_results.index.name = "模型"
df_results = df_results.reset_index()
print("\n评估结果汇总：")
print(df_results.to_string(index=False))

In [ ]:
# 打印 Markdown 格式对比表（可直接复制进 README）
print("| 模型 | HR@10 | HR@50 | Recall@50 | NDCG@50 | Coverage@100 |")
print("|------|-------|-------|-----------|---------|-------------|")
for name, r in results.items():
    print(
        f"| {name} "
        f"| {r['HR@10']:.4f} "
        f"| {r['HR@50']:.4f} "
        f"| {r['Recall@50']:.4f} "
        f"| {r['NDCG@50']:.4f} "
        f"| {r['Coverage@100']:.4f} |"
    )

## 3. 可视化对比

In [ ]:
rows_hr = []
rows_ndcg = []
for name, r in results.items():
    for k in K_LIST:
        rows_hr.append({"模型": name, "K": f"@{k}", "HR@K": r[f"HR@{k}"]})
        rows_ndcg.append({"模型": name, "K": f"@{k}", "NDCG@K": r[f"NDCG@{k}"]})

df_hr = pd.DataFrame(rows_hr)
fig_hr = px.bar(
    df_hr, x="K", y="HR@K", color="模型", barmode="group",
    title="各召回模型 HR@K 对比",
    template="plotly_white",
    labels={"HR@K": "命中率 HR@K"},
    color_discrete_sequence=px.colors.qualitative.Set2
)
_save_fig(fig_hr, "13_recall_hr.png")
fig_hr.show()

df_ndcg = pd.DataFrame(rows_ndcg)
fig_ndcg = px.bar(
    df_ndcg, x="K", y="NDCG@K", color="模型", barmode="group",
    title="各召回模型 NDCG@K 对比",
    template="plotly_white",
    labels={"NDCG@K": "归一化折损增益 NDCG@K"},
    color_discrete_sequence=px.colors.qualitative.Set2
)
_save_fig(fig_ndcg, "14_recall_ndcg.png")
fig_ndcg.show()

print("图表已保存：13_recall_hr.png, 14_recall_ndcg.png")

In [ ]:
coverage_data = {
    name: r[f"Coverage@{max(K_LIST)}"]
    for name, r in results.items()
}
df_cov = pd.DataFrame(
    [{"模型": k, "Coverage@100": v} for k, v in coverage_data.items()]
)
fig_cov = px.bar(
    df_cov, x="模型", y="Coverage@100",
    title="各召回模型 Coverage@100 对比（目录覆盖率）",
    template="plotly_white",
    color="模型",
    color_discrete_sequence=px.colors.qualitative.Set2,
    text="Coverage@100"
)
fig_cov.update_traces(texttemplate="%{text:.3f}", textposition="outside")
_save_fig(fig_cov, "15_coverage.png")
fig_cov.show()

print("图表已保存：15_coverage.png")

## 4. 多路召回融合

In [ ]:
# 实例化 MultiRecall，评估融合后效果 vs 单路最优
# 权重设计：ALS > BPR > ItemCF > TopPop
multi = MultiRecall([
    (als, 3.0),   # ALS 语义最强，权重最高
    (bpr, 2.5),   # BPR 次之
    (icf, 2.0),   # ItemCF 协同过滤
    (pop, 0.5),   # TopPop 热门兜底，权重最低
])

recs_multi = multi.recommend_batch(eval_users, k=100)
results["MultiRecall"] = evaluate_recall(
    recs_multi, ground_truth_valid, all_items, k_list=K_LIST
)

# 找单路最优基线（按 HR@50 排名）
best_single_name = max(
    [n for n in results if n != "MultiRecall"],
    key=lambda n: results[n]["HR@50"]
)
best_single = results[best_single_name]
multi_res = results["MultiRecall"]

print("\n融合效果 vs 最优单路召回器：")
print(f"{'指标':<20} {'最优单路 (' + best_single_name + ')':>20} {'MultiRecall':>15} {'增益%':>10}")
print("-" * 70)
for metric in ["HR@10", "HR@50", "Recall@50", "NDCG@50", "Coverage@100"]:
    base = best_single.get(metric, 0.0)
    multi_val = multi_res.get(metric, 0.0)
    gain = (multi_val - base) / base * 100 if base > 0 else 0.0
    print(f"{metric:<20} {base:>20.4f} {multi_val:>15.4f} {gain:>+9.2f}%")

print("\n全量指标对比表（Markdown）：")
print("| 模型 | HR@10 | HR@50 | Recall@50 | NDCG@50 | Coverage@100 |")
print("|------|-------|-------|-----------|---------|-------------|")
for name, r in results.items():
    print(
        f"| **{name}** "
        f"| {r['HR@10']:.4f} "
        f"| {r['HR@50']:.4f} "
        f"| {r['Recall@50']:.4f} "
        f"| {r['NDCG@50']:.4f} "
        f"| {r['Coverage@100']:.4f} |"
    )

## 5. 结论与调参建议

### 各模型优缺点分析

| 模型 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| **Top-Pop** | 简单高效，覆盖率高，无冷启动问题 | 不个性化，同质化严重，NDCG 低 | 新用户兜底、热门商品页 |
| **ItemCF** | 可解释性强，适合稀疏场景 | 计算复杂度高 O(I²)，新商品冷启动 | 商品详情页「相关推荐」 |
| **BPR-MF** | 成对排序损失更贴近推荐目标，效果平衡 | 超参敏感，negative sampling 影响大 | 通用召回主力 |
| **ALS** | 并行训练快，矩阵分解质量高 | 对隐式反馈需要特殊 confidence 处理 | 大规模工业召回 |
| **MultiRecall** | 多样性最高，弥补单路盲点，Coverage 最优 | 延迟略高（多路并行可缓解） | 线上主召回通道 |

### 推荐的召回策略

1. **主召回通道**：ALS + BPR-MF 加权融合（权重 3:2.5），覆盖语义相关性和排序优化
2. **辅助召回**：ItemCF 补充协同过滤信号（权重 2.0）
3. **兜底召回**：Top-Pop 对新用户/冷启动场景（权重 0.5）
4. **候选集大小**：每路各 100，融合后 Top-200 送入排序层

### 调参建议

- **BPR/ALS factors**：当前 64 维，可尝试 128/256，注意过拟合
- **ItemCF n_neighbors**：20 适合中等规模，大数据集可增加至 50
- **MultiRecall 权重**：基于验证集 HR@50 做贝叶斯超参搜索
- **候选集大小 k**：召回 k=200 → 排序 Top-20，精排阶段用 NDCG@20 评估
- **负采样策略**：BPR 中引入「热门商品更难的负采样」可进一步提升 NDCG